In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

In [49]:
def make_cv_folds(train_pool, dates, n_splits=5):
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(dates)):
        train_dates = dates[train_idx]
        val_dates = dates[val_idx]
        assert not set(train_dates) & set(val_dates)
        train_mask = train_pool['FlightDate'].isin(train_dates)
        val_mask = train_pool['FlightDate'].isin(val_dates)
        train_fold = train_pool[train_mask]
        val_fold = train_pool[val_mask]
        yield fold, train_fold, val_fold, train_dates, val_dates

In [50]:
df = pd.read_csv('../data/interim/seattle_ontime_clean.csv')
df.shape
df.columns

Index(['Unnamed: 0', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline',
       ...
       'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID',
       'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff',
       'Div5TailNum', 'Unnamed: 109'],
      dtype='object', length=111)

In [51]:

df['FlightDate'] = pd.to_datetime(df["FlightDate"])
df['FlightDate'].head()

0   2024-08-01
1   2024-08-02
2   2024-08-03
3   2024-08-04
4   2024-08-01
Name: FlightDate, dtype: datetime64[ns]

In [52]:
post_flight = [
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelayMinutes',
    'ArrDel15', 'ArrivalDelayGroups', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay',
    'DivDistance',
    'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn',
    'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum',
    'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn',
    'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum',
    'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn',
    'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum',
    'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn',
    'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum',
    'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn',
    'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum',
]
drop_cols = [
    'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepTimeBlk', 'ArrTimeBlk', 'Flights', 'DistanceGroup','Origin','Unnamed: 0', 'Unnamed: 109'
]


df = df.drop(columns=post_flight + drop_cols)
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline',
       'Dest', 'CRSDepTime', 'CRSArrTime', 'ArrDelay', 'CRSElapsedTime',
       'Distance'],
      dtype='object')

In [53]:
df['dep_minutes'] = (df['CRSDepTime'] // 100) * 60 + (df['CRSDepTime'] % 100)
df['arr_minutes'] = (df['CRSArrTime'] // 100) * 60 + (df['CRSArrTime'] % 100)
df[['CRSDepTime', 'dep_minutes', 'CRSArrTime', 'arr_minutes']].describe()

df['dep_sin'] = np.sin(2 * np.pi * df['dep_minutes'] / 1440)
df['dep_cos'] = np.cos( 2 * np.pi * df['dep_minutes'] / 1440)

df['arr_sin'] = np.sin(2 * np.pi * df['arr_minutes'] / 1440)
df['arr_cos'] = np.cos(2 * np.pi * df['arr_minutes'] / 1440)

In [54]:
cutoff = pd.Timestamp('2025-10-01')
df['DepHour'] = df['CRSDepTime'] // 100
train_pool = df[df['FlightDate'] < cutoff]
test = df[df['FlightDate'] >= cutoff]

dates = np.sort(train_pool['FlightDate'].unique())
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    print(fold, train_fold.shape[0], val_fold.shape[0], train_dates.min(), train_dates.max(), val_dates.min(), val_dates.max())


print(len(train_pool), train_pool['FlightDate'].nunique(), train_pool['FlightDate'].min(), train_pool['FlightDate'].max())
print(len(test), test['FlightDate'].nunique(), test['FlightDate'].min(), test['FlightDate'].max())
assert train_pool['FlightDate'].max() < test['FlightDate'].min()
assert len(train_pool) + len(test) == len(df)

0 41901 51554 2024-01-01T00:00:00.000000000 2024-04-18T00:00:00.000000000 2024-04-19T00:00:00.000000000 2024-08-02T00:00:00.000000000
1 93455 49684 2024-01-01T00:00:00.000000000 2024-08-02T00:00:00.000000000 2024-08-03T00:00:00.000000000 2024-11-16T00:00:00.000000000
2 143139 41431 2024-01-01T00:00:00.000000000 2024-11-16T00:00:00.000000000 2024-11-17T00:00:00.000000000 2025-03-02T00:00:00.000000000
3 184570 47184 2024-01-01T00:00:00.000000000 2025-03-02T00:00:00.000000000 2025-03-03T00:00:00.000000000 2025-06-16T00:00:00.000000000
4 231754 53895 2024-01-01T00:00:00.000000000 2025-06-16T00:00:00.000000000 2025-06-17T00:00:00.000000000 2025-09-30T00:00:00.000000000
285649 639 2024-01-01 00:00:00 2025-09-30 00:00:00
38841 92 2025-10-01 00:00:00 2025-12-31 00:00:00


In [55]:
profile_cols = ['IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Dest']
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_median = train_fold.groupby(profile_cols)['ArrDelay'].median()
    val_fold = val_fold.merge(profile_median.rename('pred').reset_index(), on=profile_cols, how='left')
    print(fold, val_fold['pred'].isna().sum(), len(val_fold))

0 16924 51554
1 9959 49684
2 6072 41431
3 11989 47184
4 5033 53895


In [56]:
mae_scores =[]
ladder_mae=[]
flat_mae =[]
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_stats = train_fold.groupby(profile_cols)['ArrDelay'].agg(['median', 'count'])
    reliable_profiles = profile_stats[profile_stats['count'] >= 10]['median']
    val_fold = val_fold.merge(reliable_profiles.rename('pred').reset_index(), on=profile_cols, how='left')
    val_fold['rung'] = np.where(val_fold['pred'].notna(),1,np.nan)


    coarse_stats = train_fold.groupby(['IATA_CODE_Reporting_Airline', 'DepHour'])['ArrDelay'].agg(['median', 'count'])
    reliable_coarse = coarse_stats[coarse_stats['count'] >= 10]['median']
    val_fold = val_fold.merge(reliable_coarse.rename('pred_coarse').reset_index(), on=['IATA_CODE_Reporting_Airline', 'DepHour'], how='left')
    unresolved = val_fold['pred'].isna()
    val_fold['pred'] = val_fold['pred'].fillna(val_fold['pred_coarse'])
    val_fold.loc[unresolved & val_fold['pred'].notna(),'rung'] = 2

    still_unresolved = val_fold['pred'].isna()
    global_median = train_fold['ArrDelay'].median()
    val_fold['pred'] = val_fold['pred'].fillna(global_median)
    val_fold.loc[still_unresolved, 'rung'] = 3
    non_rung_1 = val_fold[val_fold['rung'] != 1]
    fold_ladder_mae = (non_rung_1['ArrDelay'] - non_rung_1['pred']).abs().mean()
    ladder_mae.append(fold_ladder_mae)
    
    fold_flat_mae = (non_rung_1['ArrDelay']- global_median).abs().mean()
    flat_mae.append(fold_flat_mae) 
    fold_mae = (val_fold['ArrDelay'] - val_fold['pred']).abs().mean()
    mae_scores.append(fold_mae)
    print(fold, val_fold['rung'].value_counts().sort_index().to_dict())

print(np.mean(mae_scores), np.std(mae_scores))


print(np.mean(ladder_mae),np.std(ladder_mae))
print(np.mean(flat_mae),np.std(flat_mae))
print(np.mean(flat_mae) - np.mean(ladder_mae))

0 {1.0: 30283, 2.0: 20641, 3.0: 630}
1 {1.0: 39274, 2.0: 10143, 3.0: 267}
2 {1.0: 34109, 2.0: 7308, 3.0: 14}
3 {1.0: 34643, 2.0: 12371, 3.0: 170}
4 {1.0: 39394, 2.0: 14356, 3.0: 145}
19.951910939426405 1.209756490038425
20.42569170229878 1.433145591603604
20.84019793146333 1.3210408121534298
0.41450622916454805


In [57]:
open_cols = ['IATA_CODE_Reporting_Airline', 'Dest']
closed_cols = ['Month', 'DayOfWeek']
numeric_cols = ['DayofMonth', 'CRSElapsedTime', 'Distance', 'dep_sin', 'dep_cos', 'arr_sin', 'arr_cos']

In [58]:
ohe_open = OneHotEncoder(
    drop='first',
    handle_unknown='infrequent_if_exist',
    min_frequency=4,
)

ohe_closed = OneHotEncoder(
    drop='first',
    categories=[list(range(1, 13)), list(range(1, 8))],
)

In [59]:
#This checkes how unknown categories are handled for the airline and destination using the OneHotEncoder with infrequent categories,this actually led to me noticing that the warning says something else and the actual encoding is something else which i then traced through sklearn's codebase and eventually made a fix which got merged into main

fold0_train = next(make_cv_folds(train_pool,dates))[1]
ohe_open.fit(fold0_train[open_cols])
print(ohe_open.infrequent_categories_)
unseen = ohe_open.transform(pd.DataFrame({'IATA_CODE_Reporting_Airline': ['ZZ'], 'Dest': ['ZZZ']})).toarray()
print(unseen)
print(ohe_open.get_feature_names_out())

[None, array(['HDN'], dtype=object)]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
['IATA_CODE_Reporting_Airline_AS' 'IATA_CODE_Reporting_Airline_B6'
 'IATA_CODE_Reporting_Airline_DL' 'IATA_CODE_Reporting_Airline_F9'
 'IATA_CODE_Reporting_Airline_HA' 'IATA_CODE_Reporting_Airline_MQ'
 'IATA_CODE_Reporting_Airline_NK' 'IATA_CODE_Reporting_Airline_OO'
 'IATA_CODE_Reporting_Airline_UA' 'IATA_CODE_Reporting_Airline_WN'
 'Dest_ALW' 'Dest_ANC' 'Dest_ATL' 'Dest_AUS' 'Dest_BLI' 'Dest_BNA'
 'Dest_BOI' 'Dest_BOS' 'Dest_BUR' 'Dest_BWI' 'Dest_BZN' 'Dest_CHS'
 'Dest_CLE' 'Dest_CLT' 'Dest_CMH' 'Dest_CVG' 'Dest_DAL' 'Dest_DCA'
 'Dest_DEN' 'Dest_DFW' 'Dest_DTW' 'Dest_EUG' 'Dest_EWR' 'Dest_FAI'
 'Dest_FAT' 'Dest_FCA' 'Dest_FLL' 'Dest_GEG' 'Dest_GTF' 'Dest_HLN'
 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [60]:
preprocessor = ColumnTransformer([
    ('open', ohe_open, open_cols), #apply ohe_open to open_cols
    ('closed', ohe_closed, closed_cols),#apply ohe_closed to closed_cols
    ('num', 'passthrough', numeric_cols),#do nothing to these since linear regression can understand it since they're numeric
])

In [61]:
linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",LinearRegression())
])

print(linear_pipeline) #this is building the pipeline through which data will pass,it will first go through the preprocessor and then the linear model will evaluate it.

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('open',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='infrequent_if_exist',
                                                                min_frequency=4),
                                                  ['IATA_CODE_Reporting_Airline',
                                                   'Dest']),
                                                 ('closed',
                                                  OneHotEncoder(categories=[[1,
                                                                             2,
                                                                             3,
                                                                             4,
                                                                             5,
                                            

In [62]:
feature_cols = open_cols + closed_cols + numeric_cols
X_train_pool = train_pool[feature_cols]
y_train_pool = train_pool["ArrDelay"]

print("X shape:", X_train_pool.shape)
print("y shape:", y_train_pool.shape)

print("Missing X values:", X_train_pool.isna().sum().sum()) #checks that there are no missing values that are there across all columns
print("Missing y values:", y_train_pool.isna().sum()) #checks that no flight has a missing Arr_Delay


X shape: (285649, 11)
y shape: (285649,)
Missing X values: 0
Missing y values: 0


In [63]:
cv_splits = []# will contain each fold's training and validation for the cross_validate function
date_splitter = TimeSeriesSplit(n_splits=5) #this is a TimeSeriesSplit object which splits the dates into 5 parts and makes sure the training and validation splits are chronological as in validation doesnt have dates that are before training.

for train_date_idx,val_date_idx in date_splitter.split(dates): #gives the index for training dates and the val dates and does it per split
    train_dates = dates[train_date_idx] #retrieve the actual dates in the fold using the idx
    val_dates = dates[val_date_idx] #same as last point

    assert train_dates.max() < val_dates.min() #this is to check that the latest date in the training set is earlier than the earliest set in the validation set because if there is overlap then the model can memorise instead of predicting

    train_flight_rows = np.flatnonzero(
        train_pool["FlightDate"].isin(train_dates).to_numpy()) #what this does is basically it takes the actual training pool and checks if each row belongs to the particular dates that are in train_dates and then gives us the index of those rows since well teh cross_validate function needs the actual flight rows.
    val_flight_rows = np.flatnonzero(
        train_pool["FlightDate"].isin(val_dates).to_numpy()) #same thing as the line above but it gives the index of the rows that have the same validation dates as the dates in the val set

    cv_splits.append((train_flight_rows,val_flight_rows)) #this basically appends the indexes of the actual flight rows from the train pool and val pool to the cv_splits list,it's basically appending what are the indexes of the rows in fold 0 for the train split and the same for val_split and this runs 5 times since we have split the dates into 5 folds.
    


